# 라이브러리 로드

In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import os

import tensorflow as tf

import matplotlib.pyplot as plt

# 데이터 불러오기

In [2]:
df = pd.read_csv("./data/file_names.csv")
df.head()

,file_name,class
0,100648-1-0-0.wav,자동차 경적
1,100648-1-1-0.wav,자동차 경적
2,100648-1-2-0.wav,자동차 경적
3,100648-1-3-0.wav,자동차 경적
4,100648-1-4-0.wav,자동차 경적


In [3]:
df.sample(20)

,file_name,class
1698,S-211103_S_102_C_185_0001.mp3,도난 경보
1155,52077-3-0-8.wav,개 짖는 소리
973,24965-3-2-0.wav,개 짖는 소리
10,118496-1-1-0.wav,자동차 경적
763,176783-3-0-3.wav,개 짖는 소리
1662,S-211024_S_101_C_305_0001.mp3,응급 경보
1804,S-211107_S_103_C_015_0001.mp3,화재 경보
1163,52441-3-1-2.wav,개 짖는 소리
51,155130-1-1-0.wav,자동차 경적
1527,S-211014_S_103_C_215_0001.mp3,화재 경보


In [ ]:
# 데이터 분포 확인
df['class'].value_counts()

class
개 짖는 소리       1000
자동차 경적         563
화재 경보          186
도난 경보          130
응급 경보           67
침입 경보           21
경찰차 사이렌         18
소방차 사이렌         13
구급차 사이렌         11
공습 경보           10
가스누설 화재 경보       7
Name: count, dtype: int64

## Dataset Split
Stratified Split => Train, Test를 8:2로 나누기

In [6]:
from sklearn.model_selection import train_test_split

x, X_test, y, y_test = train_test_split(df['file_name'], df['class'], test_size=0.2, random_state=42, stratify=df['class'])
x.shape, X_test.shape, y.shape, y_test.shape

((1620,), (406,), (1620,), (406,))

In [9]:
y.value_counts()

class
개 짖는 소리       800
자동차 경적        450
화재 경보         149
도난 경보         104
응급 경보          53
침입 경보          17
경찰차 사이렌        14
소방차 사이렌        10
구급차 사이렌         9
공습 경보           8
가스누설 화재 경보      6
Name: count, dtype: int64

In [10]:
y_test.value_counts()

class
개 짖는 소리       200
자동차 경적        113
화재 경보          37
도난 경보          26
응급 경보          14
경찰차 사이렌         4
침입 경보           4
소방차 사이렌         3
공습 경보           2
구급차 사이렌         2
가스누설 화재 경보      1
Name: count, dtype: int64

## 테스트 데이터 특징 추출

In [14]:
import librosa

In [26]:
# log-mel spectrogram
def logmel_extraction(df, base_dir, mp3=True, sr=16000, n_fft=1024, hop_length=512, n_mels=64, max_len=400):
    X = []
    y = []

    for fname, label in tqdm(zip(df['file_name'], df['class']), total=len(df), desc='Extracting Log-Mel'):
        # 실제 파일 경로 만들기
        path = os.path.join(base_dir, fname)

        # Load audio
        data_x, sampling_rate = librosa.load(path, sr=sr)

        # mp3 파일은 앞 9초 제거
        if fname.endswith('.mp3') and mp3:
            data_x = data_x[9 * sr:]

        # Mel Spectrogram
        mel = librosa.feature.melspectrogram(
            y=data_x,
            sr=sampling_rate,  # sampling rate
            n_fft=n_fft,       # fft window size
            hop_length=hop_length,  # fram간 이동 간격
            n_mels=n_mels,     # mel 필터 수
            fmin=20,           # 분석할 최소 주파수
            fmax=sampling_rate // 2  # 분석할 최대 주파수
        )

        # Log scaling (dB)
        logmel = librosa.power_to_db(mel, ref=np.max).astype(np.float32)
        
        # 길이 맞추기
        if logmel.shape[1] < max_len:
            pad_width = max_len - logmel.shape[1]
            logmel = np.pad(logmel, ((0, 0), (0, pad_width)), 'constant')
        else:
            logmel = logmel[:, :max_len]

        X.append(logmel[np.newaxis, :, :])    # (n_mels, time)
        y.append(label)

    return np.array(X), np.array(y)

In [27]:
# mfcc
def mfcc_extraction(df, base_dir, mp3=True, n_mfcc=50, sr=16000, max_len=400):
    X = []
    y = []

    for fname, label in tqdm(zip(df['file_name'], df['class']), total=len(df), desc='Extracting MFCC'):
        # 실제 파일 경로 만들기
        path = os.path.join(base_dir, fname)

        # Load audio
        data_x, sampling_rate = librosa.load(path, sr=sr)

        # mp3는 앞 9초 제거
        if fname.endswith('.mp3') and mp3:
            data_x = data_x[9 * sr:]  # 9초 * sampling rate

        # MFCC 추출
        mfcc = librosa.feature.mfcc(
            y=data_x,
            sr=sampling_rate,
            n_mfcc=n_mfcc,      # 추출할 MFCC 수
            n_fft=1024,         # STFT 윈도우 크기
            hop_length=512
        ).astype(np.float32)
        
        if mfcc.shape[1] < max_len:
            pad_width = max_len - mfcc.shape[1]
            mfcc = np.pad(mfcc, ((0, 0), (0, pad_width)), 'constant')
        else:
            mfcc = mfcc[:, :max_len]

        X.append(mfcc[np.newaxis, :, :])
        y.append(label)

    return np.array(X), np.array(y)

In [20]:
test_df = pd.DataFrame({'file_name':X_test, 'class':y_test})
base_dir = './data/Train'

In [22]:
# log-mel
X_test_logmel, y_test_logmel = logmel_extraction(test_df, base_dir)

Extracting Log-Mel:  38%|███▊      | 153/406 [00:43<00:14, 17.01it/s]C:\Users\leehb\AppData\Roaming\Python\Python313\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=1024 is too large for input signal of length=960
  warnings.warn(
Extracting Log-Mel: 100%|██████████| 406/406 [01:04<00:00,  6.31it/s]


In [23]:
X_test_logmel.shape, y_test_logmel.shape

((406, 1, 64, 400), (406,))

In [24]:
# mfcc
X_test_mfcc, y_test_mfcc = mfcc_extraction(test_df, base_dir)

Extracting MFCC:  37%|███▋      | 149/406 [00:06<00:12, 20.17it/s]C:\Users\leehb\AppData\Roaming\Python\Python313\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=1024 is too large for input signal of length=960
  warnings.warn(
Extracting MFCC: 100%|██████████| 406/406 [00:15<00:00, 26.45it/s]


In [25]:
X_test_mfcc.shape, y_test_mfcc.shape

((406, 1, 50, 400), (406,))

# Stratified K-Fold

In [ ]:
from sklearn.model_selection import StratifiedKFold, train_test_split
from tensorflow.keras import layers, models

In [ ]:
# 증강 함수


In [ ]:
# CNN 모델 정의


In [ ]:
# Stratified K-Fold 
def stratified_kfold_training(x, y, base_dir, feature_extraction, n_splits=5, epochs=20, batch_size=32, sr=16000, n_mels=64, max_len=400, n_mfcc=50):
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    num_clases = len(np.unique(y))
    
    results = []
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(x, y)):
        print(f'\n===== Fold {fold+1} =====')
        
        train_df = pd.DataFrame({'file_name': x.iloc[train_idx], 'class': y.iloc[train_idx]})
        valid_df = pd.DataFrame({'file_name': x.iloc[val_idx], 'class': y.iloc[val_idx]})
        
        # train 증강 + 특징 추출
        aug_files = []
        aug_labels = []
        for fname, label in zip(train_df['file_name'], train_df['class']):
            path = os.path.join(base_dir, fname)
            audio, sr = librosa.load(path, sr=sr)
            
            if fname.endswith('.mp3'):
                audio = audio[9*sr:]
            
            # 원본
            aug_files.append(fname)
            aug_labels.append(label)
            
            # 증강된 오디오
            aug_audio = augmentation(audio)
            aug_files.append(fname)
            aug_labels.append(label)
            
        aug_df = pd.DataFrame({'file_name': aug_files, 'class': aug_labels})
        
        if feature_extraction == 'logmel':
            X_train, y_train = logmel_extraction(aug_df, base_dir, mp3=False)
            X_val, y_val = logmel_extraction(valid_df, base_dir, mp3=False)
        elif feature_extraction == 'mfcc':
            X_train, y_train = mfcc_extraction(aug_df, base_dir, mp3=False)
            X_val, y_val = mfcc_extraction(valid_df, base_dir, mp3=False)
        else:
            raise ValueError('feature_extraction must be 'logmel' or 'mfcc'')
        
        # 모델 정의 및 컴파일
        input_shape = X_train.shape[1:]
        model = build_cnn(input_shape, num_clases)
        
        model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        
        # 학습
        history = model.fit(X_train, y_train, 
                            epochs=epochs, 
                            batch_size=batch_size, 
                            verbose=1, 
                            validation_data=(X_val, y_val))
        
        # 모델 저장
        save_path = f'./models/model_fold{fold+1}.h5'
        model.save(save_path)
        print(f'Model saved at {save_path}')
        
        # 평가
        
        
    return results